# 📊 Sistema de Monitoreo de Métricas para Grafana

Este notebook demuestra cómo usar el sistema de monitoreo de métricas de trading para enviar datos a Grafana.

## Características

- Monitoreo automático de todos los sistemas o sistemas específicos
- Obtención dinámica del run más reciente por sistema (basado en `started_at`)
- Cálculo y acumulación de métricas de PnL
- Generación de métricas agregadas para `ALL_SYSTEMS`
- Refresco opcional de sistemas dinámicamente

In [ ]:
import asyncio

from logging_system import setup_logging
from copy_trading.grafana import TradingMetricsMonitor

setup_logging(
    console_output=True,
    file_output=False,
    log_directory="logs",
    log_filename="grafana_monitoring_%Y-%m-%d_%H-%M-%S.log",
    min_level_to_process="DEBUG",
    enable_logfire=False,
)

## Monitoreo en producción (loop infinito)

Ejemplo de cómo ejecutar el monitor de forma continua en producción.

In [ ]:
async def production_monitoring():
    """
    Ejemplo de monitoreo en producción.
    
    Este ejemplo muestra cómo ejecutar el monitor de forma continua,
    con manejo de errores y logging apropiado.
    """
    monitor = TradingMetricsMonitor()

    try:
        # Iniciar monitoreo
        await monitor.start(
            interval_seconds=10,           # Verificar nuevos datos cada 10s
            periodic_write_seconds=60,     # Escribir métricas cada 60s
            refresh_systems=False,          # Refrescar sistemas dinámicamente
            refresh_systems_interval_seconds=300,  # Refrescar cada 5 minutos
            delete_previous_metrics=True,   # Borrar todos los datos de la tabla TradingMetrics al iniciar
        )

        print("🚀 Monitor iniciado en modo producción")
        print("📊 Monitoreando todos los sistemas...")
        print("⏹️  Presiona stop para detener")

        # Loop infinito (en producción, esto podría estar en un servicio)
        while True:
            await asyncio.sleep(60)  # Esperar 1 minuto entre verificaciones

    except KeyboardInterrupt:
        print("\n🛑 Señal de interrupción recibida...")
    except asyncio.CancelledError:
        print("\n🛑 Monitor cancelado")
    except Exception as e:
        print(f"\n❌ Error en el monitor: {e}")
        import traceback
        traceback.print_exc()
    finally:
        print("🔄 Deteniendo monitor...")
        await monitor.stop()
        print("✅ Monitor detenido correctamente")

await production_monitoring()

## Métricas generadas

El sistema genera las siguientes métricas en la tabla `TradingMetrics`:

### Por sistema individual:
- **PNL_Individual_Cumulative**: PnL acumulado por trader individual
- **PNL_Total_Cumulative**: PnL total acumulado del sistema
- **Capital_Current**: Capital actual (pendiente de implementar)
- **Capital_Total**: Capital total (pendiente de implementar)
- **Drawdown_Individual**: Drawdown por trader (pendiente de implementar)
- **Drawdown_Total**: Drawdown total (pendiente de implementar)
- **Return_Total_Percentage**: Retorno total en porcentaje (pendiente de implementar)

### Agregadas (ALL_SYSTEMS):
- **PNL_Total_Cumulative**: Suma del PnL de todos los sistemas

### Campos de cada métrica:
- `id`: UUID único
- `timestamp`: Timestamp de la métrica
- `metric_name`: Nombre de la métrica
- `metric_value`: Valor numérico (en SOL)
- `system_name`: Nombre del sistema o "ALL_SYSTEMS"
- `execution_mode`: "live" o "dry_run"
- `trader`: Dirección del trader o "ALL_TRADERS"

## Notas importantes

1. **Runs dinámicos**: El monitor obtiene automáticamente el run más reciente de cada sistema basándose en `started_at`. Si un sistema tiene un nuevo run, el monitor lo detectará (si `refresh_systems=True`).

2. **Procesamiento incremental**: El sistema solo procesa nuevos datos de PnL desde la última métrica guardada, evitando reprocesar datos antiguos.

3. **Métricas agregadas**: Las métricas de `ALL_SYSTEMS` se generan automáticamente sumando el PnL total de todos los sistemas monitoreados.

4. **Modos de ejecución**: Las métricas se separan por `execution_mode` (live/dry_run) para mantener datos independientes.

5. **Persistencia**: Todas las métricas se guardan en la tabla `TradingMetrics` y pueden ser consultadas posteriormente o enviadas a Grafana.